# 11 · The Left CLI: config, options, checks, tools and `dtrack dbx`

This notebook drives the real `dtrack` click CLI (`dtrack_left.cli`) in-process through `dtrack_left.cli.main(argv)`, the same entry point as the `dtrack` console script. It uses your real config, a real S3 scratch prefix and your Databricks workspace. Each command prints one JSON line, and the notebook shows it together with the exit code.

Mirrors `tests/unit/dtrack_left/cli/test_config.py`, `test_options.py`, `test_checks.py`, `test_tools.py`, `tests/unit/dtrack_left/cli/ops/test_s3.py` and `ops/test_dbx.py`. The tests monkeypatch the backends. Here the commands reach the real Left source, S3 and Databricks. Anything that would change Databricks or your AWS files is gated by `DRY_RUN`.

## How to approach

**Prerequisites**
- Kernel: the repo `.venv` (`uv sync --extra dbx`, plus `--extra athena` / `--extra ldap` if a pair needs them).
- `ENV_FILE` is loaded first. The CLI itself only searches for `.env` from the current directory upward, and the notebook's directory is not the repo.
- AWS credentials that can write `S3_SCRATCH`. The AWS profile `AWS_PROFILE` is used for `dbx s3 whoami`.
- `~/.databrickscfg` profile `DBX_PROFILE`, a SQL warehouse id, and optionally a cluster id.

**Run order.** Run it top to bottom.
- Sections 1-3 are offline apart from one S3 listing.
- Section 4 (`doctor`) preflights every active pair, so it is slow when you have many.
- Sections 5-7 run a real `row-check` / `col-check` / `fetch-report` cycle for one pair inside the scratch prefix. The Right row check is played through the SQL warehouse, as in notebook 10.
- Section 8 (`rollback`) is a dry run plus a harmless `--stage dead --yes`.
- Sections 9-10 are the operator helpers. Reads always run. `warehouse start`, a real workspace upload and `s3 creds` only run with `DRY_RUN = False`.

**What to look at.** Exit codes: 0 means ok, 1 means a command or backend failure (e.g. doctor found a broken source), 2 means a usage error. Look at the JSON payloads and at the env-var fallbacks (`DTRACK_S3`, `DTRACK_DBX_*`).

**Failure modes**
- Exit 2 on every command: a path or option is wrong, and the click message is shown above the JSON.
- `{"error": "ImportError ..."}`: install `dtrack[dbx]`.
- `{"error": "...ProfileNotFound"}`: the AWS/Databricks profile name is wrong.
- `doctor` exit 1: at least one pair's Left preflight failed, and its `error` says why.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# The project .env first, so the DTRACK_* defaults below and the Left credentials resolve
ENV_FILE = os.environ.get("DTRACK_ENV_FILE", "C:/work/dtrack/.env")   # repo .env on this machine
load_dotenv(ENV_FILE, override=False)

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")   # your real pairs config
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")             # the pair used for row-check / col-check
TO_DATE = None                                                 # None = yesterday; or "YYYY-MM-DD"
DAYS = 3                                                       # window = the last DAYS days up to TO_DATE
S3_SCRATCH = os.environ.get("DTRACK_PROD_S3", "s3://<bucket>/dtrack/nb-scratch")   # the only S3 writes go here
REPORT_S3 = os.environ.get("DTRACK_REPORT_S3", "")            # your real report store: read-only fetch; "" = skip
DBX_PROFILE = os.environ.get("DTRACK_DBX_PROFILE", "corp")    # ~/.databrickscfg profile
WAREHOUSE_ID = os.environ.get("DTRACK_DBX_WAREHOUSE_ID", "")  # SQL warehouse id
CLUSTER_ID = os.environ.get("DTRACK_DBX_CLUSTER_ID", "")      # all-purpose cluster id; "" = skip compute status
WORKSPACE_TARGET = None                                        # None = /Users/<me>/dtrack_right for the upload
AWS_PROFILE = os.environ.get("DTRACK_AWS_PROFILE", "corp")    # ~/.aws/config source profile
AWS_TARGET = os.environ.get("DTRACK_AWS_TARGET", "corp-winscp")   # static profile `dbx s3 creds` writes
DISPOSITION = os.environ.get("DTRACK_LOCAL_DISPOSITION", "external_links")   # warehouse result transport
WAIT_TIMEOUT = 600                                             # col-check --wait-timeout (seconds)
MAX_ROWS = 200_000                                             # refuse a col-check bigger than this many Left rows
LOG_LEVEL = "INFO"                                             # passed as --log-level to every command
WORKDIR = Path.home() / ".local/plans/nbs/_work/11"           # window config, pulls, fetched reports
KEEP = False                                                   # True = keep the scratch prefix
DRY_RUN = True                                                 # False = really start the warehouse, upload, write ~/.aws/credentials

In [ ]:
import copy
import json
from datetime import date, timedelta

import pandas as pd

from dtrack_left.checks.row_check import make_run_id
from dtrack_left.cli.config import load_config
from dtrack_left.cli.options import setup_logging

# dtrack logs (UTC timestamps) go to stderr, which Jupyter shows under each cell
setup_logging(LOG_LEVEL)

# Load and validate the real config: conn_macro profiles merged, col_map files expanded
config = load_config(CONFIG_PATH)
active = [name for name, row in config["pairs"].items() if not row.get("skip")]
print(f"{len(active)} active pair(s): {active}")
if PAIR not in active:
    raise ValueError(f"PAIR={PAIR!r} is not an active pair of {CONFIG_PATH}")

# The check window: the last DAYS days up to TO_DATE (default yesterday), both ends inclusive
TO = TO_DATE or (date.today() - timedelta(days=1)).isoformat()
FROM = (date.fromisoformat(TO) - timedelta(days=DAYS - 1)).isoformat()

# A deep copy of the pair with its dates narrowed to the window; the real config is never changed
pair = copy.deepcopy(config["pairs"][PAIR])
pair["fromDate"], pair["toDate"] = FROM, TO
run_config = {"pairs": {PAIR: pair}}
DATE_COL = pair["left"]["date_col"]
KEYS = [k.strip() for k in str(pair.get("key") or "").split(",") if k.strip()]
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"pair {PAIR}: {pair['left'].get('source')} -> databricks, window {FROM}..{TO}, key={KEYS or None}")
print(f"col_map: {len(pair['col_map'])} column(s), sample={pair.get('sample')!r}, vintage={pair.get('vintage') or 'day'}")

import contextlib
import io

from dtrack_left.backends.store import S3Store
from dtrack_left.cli import main

RUN_ID = make_run_id(run_config)
NB_ROOT = f"{S3_SCRATCH.rstrip('/')}/nb11-{RUN_ID}"
RUN_URI = f"{NB_ROOT}/runs/{RUN_ID}"
REPORT_URI = f"{NB_ROOT}/report"

# The CLI reads --config from a file: write the window-narrowed pair next to the other local files.
# Note: this copy holds whatever the real config holds for this pair (conn fields included); it stays local.
WINDOW_CONFIG = WORKDIR / "c_window.json"
WINDOW_CONFIG.write_text(json.dumps(run_config, indent=2))


# Run one dtrack command in-process: exit code + its one JSON line (help text is printed as-is)
def dtrack(*argv, log=True):
    args = [str(item) for item in argv]
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        code = main(args)
    text = buffer.getvalue().strip()
    print(f"$ dtrack {' '.join(args)}\n-> exit {code}")
    try:
        payload = json.loads(text.splitlines()[-1]) if text else None
    except ValueError:
        payload = None
    print(json.dumps(payload, indent=2, default=str)[:4000] if payload is not None else text)
    return code, payload


print(f"RUN_URI = {RUN_URI}\nWINDOW_CONFIG = {WINDOW_CONFIG}")

## 1. Help, version and usage errors

`main` returns the command's exit code instead of calling `sys.exit`. Click usage errors map to 2, `--version` / `-h` to 0, and the group lists every command. None of these cells touch a system.

In [ ]:
# Top-level help lists row-check, col-check, fetch-report, doctor, rollback, debug, dbx
dtrack("-h")
dtrack("--version")

# Usage errors exit 2 without running anything: a missing config file, debug without a command
dtrack("row-check", "--config", WORKDIR / "nope.json", "--s3", RUN_URI)
dtrack("debug")

## 2. The config file

`load_config` loads `.env`, expands `col_map` files (CSV `left,right[,type]` or JSON, with paths relative to the config file) and merges `connections` profiles into any side with a matching `conn_macro`. `validate_config` then enforces the rules: a supported Left source, a Databricks Right side, a known Right kind with its required fields, `date_col` on both sides, a non-empty `col_map`, and valid `col_type_overrides`. The last cell breaks copies of your real pair to show each error message.

In [ ]:
from dtrack_left.cli.config import LEFT_SOURCES, RIGHT_KINDS

# One row per pair of the real config as dtrack sees it after load_config
rows = []
for name, row in config["pairs"].items():
    left, right = row.get("left") or {}, row.get("right") or {}
    rows.append({
        "pair": name, "skip": bool(row.get("skip")),
        "left": left.get("source"), "left_macro": left.get("conn_macro"), "left_date": left.get("date_col"),
        "right_kind": right.get("kind") or "table", "right_date": right.get("date_col"),
        "columns": len(row.get("col_map") or {}), "overrides": len(row.get("col_type_overrides") or {}),
        "sample": row.get("sample"), "vintage": row.get("vintage") or "day", "key": row.get("key"),
    })
print("Left sources:", sorted(LEFT_SOURCES), "| Right kinds:", {k: v for k, v in RIGHT_KINDS.items()})
pd.DataFrame(rows)

In [ ]:
from dtrack_left.cli.config import read_col_map

# Pairs whose raw config points col_map at a file: show what that file expands to
raw = json.loads(Path(CONFIG_PATH).read_text(encoding="utf-8"))
base = Path(CONFIG_PATH).resolve().parent
files = {name: row["col_map"] for name, row in raw["pairs"].items() if isinstance(row.get("col_map"), str)}
print("col_map files:", files or "none (all inline)")
for name, relative in files.items():
    mapping, types = read_col_map(base / relative)
    print(f"{name}: {len(mapping)} column(s) from {relative}; types from the file: {types or 'none'}")
print("connections profiles:", sorted(raw.get("connections") or {}) or "none")

In [ ]:
from dtrack_left.cli.config import validate_config

# Each change breaks a copy of the real pair in one way; validate_config names the problem
breakers = {
    "Right side not databricks": lambda p: p["right"].update(source="csv"),
    "Left source out of scope": lambda p: p["left"].update(source="sas"),
    "no Right date_col": lambda p: p["right"].pop("date_col"),
    "empty col_map": lambda p: p.update(col_map={}),
    "unknown Right kind": lambda p: p["right"].update(kind="view"),
    "query kind without query": lambda p: (p["right"].update(kind="query"), p["right"].pop("query", None)),
    "{src} outside files kind": lambda p: p["right"].update(kind="table", query="SELECT * FROM {src}"),
    "override on unmapped column": lambda p: p.update(col_type_overrides={"dtrack_nope": "numeric"}),
}
for label, change in breakers.items():
    candidate = copy.deepcopy(pair)
    change(candidate)
    try:
        validate_config({"pairs": {PAIR: candidate}})
        print(f"{label:30} accepted")
    except (ValueError, TypeError) as exc:
        print(f"{label:30} {type(exc).__name__}: {exc}")

## 3. Shared options: `--s3`, `--pair`, `--config`, `--log-level`, one JSON line

Every command is built from the same decorators in `dtrack_left.cli.options`. `store_option` turns `--s3` (or `DTRACK_S3`) into an `S3Store`, and `config_option` loads and validates `--config`. `pair_option` collects repeated `--pair`, `log_level_option` configures stderr logging before the body runs, and `emit_json` prints the return dict and turns its `exit_code` into the exit code. A throwaway probe command shows all of them against the real scratch prefix.

In [ ]:
import click
from click.testing import CliRunner

from dtrack_left.cli import options


# A probe wired with every shared decorator; it lists the run folder to prove the store is live
@click.command(context_settings=options.CONTEXT_SETTINGS)
@options.config_option
@options.store_option()
@options.pair_option
@click.option("--code", default=0)
@options.log_level_option
@options.emit_json
def probe(config, store, pairs, code):
    return {
        "store": type(store).__name__, "uri": store.root_uri, "objects": len(store.list_keys("")),
        "pairs": pairs, "config_pairs": sorted(config["pairs"]), "exit_code": code,
    }


runner = CliRunner()
result = runner.invoke(probe, ["--config", str(WINDOW_CONFIG), "--s3", RUN_URI, "--pair", PAIR, "--pair", "x"], standalone_mode=False)
print("flags  :", result.output.strip(), "| exit", result.return_value)

# The same options from environment variables (DTRACK_S3, DTRACK_CONFIG); the exit_code key becomes the exit code
result = runner.invoke(probe, ["--code", "3", "--log-level", "warning"], env={"DTRACK_S3": RUN_URI, "DTRACK_CONFIG": str(WINDOW_CONFIG)}, standalone_mode=False)
print("env    :", result.output.strip(), "| exit", result.return_value)

## 4. `dtrack doctor` and `describe_right`

`doctor` preflights the Left source of every active pair (`SELECT * ... WHERE 1=0`). It also shows what the Databricks job will read for the Right side, which the Left side never connects to. The exit code is 1 if any Left preflight fails.

In [ ]:
from dtrack_left.cli import tools

# Every active pair of the REAL config file (not just PAIR): slow with many pairs
code_, doctor = dtrack("doctor", "--config", CONFIG_PATH, "--log-level", LOG_LEVEL)
pd.DataFrame(doctor).T if isinstance(doctor, dict) else doctor

In [ ]:
# What the Right job will read for each pair: kind, target relation, and whether a {src} query is patched in
pd.DataFrame({name: tools.describe_right(row["right"]) for name, row in config["pairs"].items() if not row.get("skip")}).T

## 5. `dtrack row-check` into the scratch run

The run id is always the last folder of `--s3`. A different `--run-id` is a usage error, and so is an `--s3` with no run folder. The real command counts the pair's Left side for the window and writes `row_check/<pair>.left.json` plus `_LEFT_CLOSED`.

In [ ]:
from dtrack_left.backends.store import S3Location

# Both refusals happen before any counting (exit 2)
bucket = S3Location.parse(S3_SCRATCH).bucket
dtrack("row-check", "--config", WINDOW_CONFIG, "--s3", RUN_URI, "--run-id", "something-else")
dtrack("row-check", "--config", WINDOW_CONFIG, "--s3", f"s3://{bucket}")

# The real thing: one pair, one worker, debug logging shows the count SQL
dtrack("row-check", "--config", WINDOW_CONFIG, "--s3", RUN_URI, "--pair", PAIR, "--workers", 1, "--log-level", LOG_LEVEL)
store = S3Store(RUN_URI)
left_doc = store.get_json(f"row_check/{PAIR}.left.json")
print("_LEFT_CLOSED:", store.get_json("row_check/_LEFT_CLOSED"), "| days:", len(left_doc["counts"]), "| rows:", sum(left_doc["counts"].values()))

## 6. `dtrack col-check` after the Right side releases manifests

`col-check` waits for `row_check/_CLOSED`, publishes fixed-N keys if needed, then publishes every unit. Here the Right row check runs through the SQL warehouse first (`dtrack_right` + `dtrack_local.WarehouseTarget`, read-only), standing in for the Databricks job. `DTRACK_S3` is set so `--s3` comes from the environment, as in the unit test.

In [ ]:
from dtrack_left.backends.databricks import workspace
from dtrack_local.pipeline import WarehouseTarget
from dtrack_local.warehouse import Warehouse
from dtrack_right.checks.row_check import run_row_check as right_row_check

# The Right side's row check, here, through the warehouse: writes the verdict, manifest and row_check/_CLOSED
warehouse = Warehouse(workspace(DBX_PROFILE), WAREHOUSE_ID, disposition=DISPOSITION)
right_row_check(store, lambda right: WarehouseTarget(warehouse, right, WORKDIR / "right"))
verdict = store.get_json(f"row_check/{PAIR}.json")
print("summary:", verdict["summary"], "| gate passed:", verdict["gate_decision"]["passed"])

# Guard before the Left pull
manifest = store.get_json(f"manifest/{PAIR}.json") if store.exists(f"manifest/{PAIR}.json") else None
if manifest and sum(w["n_left"] for w in manifest["windows"]) > MAX_ROWS:
    raise RuntimeError("released windows exceed MAX_ROWS: lower DAYS or raise MAX_ROWS")

In [ ]:
# --s3 from DTRACK_S3, exactly like the unit test; the command blocks until manifests are released
os.environ["DTRACK_S3"] = RUN_URI
try:
    dtrack("col-check", "--pair", PAIR, "--work-dir", WORKDIR / "col_check", "--wait-timeout", WAIT_TIMEOUT,
           "--poll-interval", 5, "--pull-workers", 1, "--stat-workers", 2, "--wip", 200, "--log-level", LOG_LEVEL)
finally:
    os.environ.pop("DTRACK_S3", None)
print("manifest/_CLOSED:", store.get_json("manifest/_CLOSED") if store.exists("manifest/_CLOSED") else "missing")
print("_STATUS.json    :", store.get_json("_STATUS.json"))

## 7. `dtrack fetch-report`

`fetch-report` downloads `history/<run_id>/` from a report store. Without `--run-id`, it follows `latest/pointer.json`. First a report for the scratch run is built and published into the scratch prefix with the Right side's own `build_report` / `publish_report`, then fetched back. If `REPORT_S3` is set, the latest real report is also fetched. That second fetch only reads.

In [ ]:
from dtrack_right.backends.store import open_store
from dtrack_right.report import build_report, publish_report

# Build the scratch run's report locally and publish it to the scratch report store
files = build_report(open_store(RUN_URI), WORKDIR / "report_build")
summary = json.loads(files["summary.json"].read_text())
print("published:", publish_report(open_store(REPORT_URI), RUN_ID, files, summary))

# Fetch it back through the CLI: the latest pointer, then an explicit --run-id
dtrack("fetch-report", "--report-s3", REPORT_URI, "--out", WORKDIR / "fetched_latest")
dtrack("fetch-report", "--report-s3", REPORT_URI, "--out", WORKDIR / "fetched_by_id", "--run-id", RUN_ID)

# Your real report store, read-only
if REPORT_S3:
    dtrack("fetch-report", "--report-s3", REPORT_S3, "--out", WORKDIR / "fetched_real")

## 8. `dtrack rollback` and `dtrack debug`

`rollback` without `--yes` only prints the plan and a `dry_run` note. It works on S3 run folders and on local ones. `--stage dead --yes` on this fresh run deletes nothing unless a unit died. `debug` re-runs any command under ipdb, or waits for a VS Code attach. It is interactive, so run it from a terminal (command printed below).

In [ ]:
# Dry runs on the scratch run: col redoes Stage 2, row redoes Stage 1 and 2
dtrack("rollback", "--run", RUN_URI, "--pair", PAIR, "--stage", "col")
dtrack("rollback", "--run", RUN_URI, "--pair", PAIR, "--stage", "row")

# Applied, but only dead units (normally none): the scratch run is otherwise untouched
dtrack("rollback", "--run", RUN_URI, "--pair", PAIR, "--stage", "dead", "--yes")

# A col rollback with a config refuses Stage 1 changes; WINDOW_CONFIG is the config that was counted, so it passes the check
dtrack("rollback", "--run", RUN_URI, "--pair", PAIR, "--stage", "col", "--config", WINDOW_CONFIG)

print("\nin a terminal:  dtrack debug -- doctor --config", CONFIG_PATH)
print("VS Code attach: dtrack debug --attach -- doctor --config", CONFIG_PATH, "  (port 5678)")

## 9. `dtrack dbx`: Databricks operator helpers

Each subcommand gets a `WorkspaceClient` for `--profile` (default from `DTRACK_DBX_PROFILE`). SDK, auth and timeout errors become `{"error": ...}` with exit 1. Reads run as-is. `warehouse start` and a real `workspace upload` only run with `DRY_RUN = False`.

In [ ]:
import dtrack_right
from dtrack_right.backends.targets import SparkTarget

# Who am I (no browser), warehouse state, and optionally the cluster state
dtrack("dbx", "login", "--no-open", "--profile", DBX_PROFILE)
dtrack("dbx", "warehouse", "status", "--warehouse-id", WAREHOUSE_ID, "--profile", DBX_PROFILE)
if CLUSTER_ID:
    dtrack("dbx", "compute", "status", "--cluster-id", CLUSTER_ID, "--profile", DBX_PROFILE)

# Ids and profile fall back to DTRACK_DBX_* (from .env): no --warehouse-id, no --profile
dtrack("dbx", "warehouse")

# One read-only statement: the Right relation's columns as the warehouse spells them
relation = SparkTarget(None, pair["right"]).relation()
dtrack("dbx", "sql", f"DESCRIBE QUERY SELECT * FROM {relation}", "--warehouse-id", WAREHOUSE_ID, "--limit", 200)
sql_file = WORKDIR / "q.sql"
sql_file.write_text("SELECT current_user() AS me, current_catalog() AS catalog, current_timestamp() AS now")
dtrack("dbx", "sql", "--file", sql_file, "--warehouse-id", WAREHOUSE_ID)

# Latest job runs, then the detail of the newest one
_, runs = dtrack("dbx", "runs", "--limit", 5)
if runs and runs.get("runs"):
    dtrack("dbx", "runs", "--run-id", runs["runs"][0]["run_id"])

In [ ]:
# Upload src/dtrack_right to the Workspace: always a --dry-run first (planned remote paths only)
RIGHT_PACKAGE = Path(dtrack_right.__file__).parent
target_args = [WORKSPACE_TARGET] if WORKSPACE_TARGET else []
dtrack("dbx", "workspace", "upload", RIGHT_PACKAGE, *target_args, "--profile", DBX_PROFILE, "--dry-run")

# Writes to Databricks: only with DRY_RUN = False
if DRY_RUN:
    print("DRY_RUN: would run  dtrack dbx workspace upload", RIGHT_PACKAGE, *target_args, "--overwrite")
    print("DRY_RUN: would run  dtrack dbx warehouse start --warehouse-id", WAREHOUSE_ID, "--wait")
else:
    dtrack("dbx", "workspace", "upload", RIGHT_PACKAGE, *target_args, "--profile", DBX_PROFILE, "--overwrite")
    dtrack("dbx", "warehouse", "start", "--warehouse-id", WAREHOUSE_ID, "--wait", "--timeout", 20)

# A usage error (no statement) is exit 2; a bad id is a backend error, exit 1 with {"error"}
dtrack("dbx", "sql", "--warehouse-id", WAREHOUSE_ID)
dtrack("dbx", "warehouse", "status", "--warehouse-id", "dtrack-no-such-warehouse")

## 10. `dtrack dbx s3`: AWS credentials for WinSCP and friends

`whoami` resolves the `--profile` credentials (credential_process, SSO or static) and shows the account, ARN and expiry. `creds` copies them into the static `--target` profile in `~/.aws/credentials` for tools that cannot run a credential process. It refuses to overwrite the source profile, and here it only runs with `DRY_RUN = False`. `winscp` also opens a GUI and keeps refreshing, so the command is printed for a terminal instead.

In [ ]:
# Read-only: identity and expiry of the source profile
dtrack("dbx", "s3", "-h")
dtrack("dbx", "s3", "whoami", "--profile", AWS_PROFILE)

# Writes ~/.aws/credentials [AWS_TARGET]: only with DRY_RUN = False (never with --target = the source profile)
if DRY_RUN:
    print(f"DRY_RUN: would run  dtrack dbx s3 creds --profile {AWS_PROFILE} --target {AWS_TARGET}")
else:
    dtrack("dbx", "s3", "creds", "--profile", AWS_PROFILE, "--target", AWS_TARGET)
print(f"in a terminal (Windows): dtrack dbx s3 winscp --profile {AWS_PROFILE} --target {AWS_TARGET}")

## Cleanup

This deletes everything under this notebook's scratch root (`NB_ROOT`: the run and the scratch report store) and the local `WORKDIR`, unless `KEEP = True`.

In [ ]:
import shutil

from dtrack_left.checks import rollback

# Remove exactly what this notebook created
if KEEP:
    print("kept", NB_ROOT)
else:
    scratch = S3Store(NB_ROOT)
    print("deleted", rollback.delete(scratch, scratch.list_keys("")), "object(s) under", NB_ROOT)
    shutil.rmtree(WORKDIR, ignore_errors=True)

## What to check

- Section 1: `-h` lists every command, and usage errors return 2 without side effects.
- Section 2: each pair shows the source, kind and column count you expect. `col_map` files expand, and every broken copy fails with a clear message.
- Section 3: flags and `DTRACK_S3` / `DTRACK_CONFIG` give the same store and config, and `exit_code` passes through.
- Section 4: `doctor` exit 0, with every pair `ok`. `describe_right` names the relation the job will read.
- Sections 5-6: `row-check` prints `{"pairs": 1, "run_id": RUN_ID}`. `col-check` prints `published_units` equal to `manifest/_CLOSED.expected_units`.
- Section 7: both fetches download `index.html`, `summary.json` and the CSVs into `WORKDIR`.
- Section 8: dry runs print `dry_run`, `--stage dead --yes` deletes 0, and the config check passes.
- Sections 9-10: login, warehouse status, SQL, runs and whoami succeed. The upload dry-run lists `.../dtrack_right/notebook` among the planned paths.